# 🚀 AI 安全模型训练 - Qwen2.5-7B

**公司**: 杭州市上城区乐友信息服务工作室

---

## 第一步: 验证 GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

## 第二步: 克隆代码并安装依赖

In [ ]:
!git clone https://github.com/richard3153/nexify-landing.git

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets loralib huggingface_hub scipy

---

## 第三步: 下载训练数据

In [ ]:
import os
os.makedirs("nexify-landing/data", exist_ok=True)

from datasets import load_dataset
import json

print("📥 下载 BeaverTails 数据集...")
beavertails = load_dataset("PKU-Alignment/BeaverTails", split="train[:3000]")
print(f"✅ 共 {len(beavertails)} 条")

records = []
for i, item in enumerate(beavertails):
    records.append({
        "id": i,
        "instruction": item["prompt"],
        "output": item["response"]
    })

with open("nexify-landing/data/train.json", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"💾 保存完成: nexify-landing/data/train.json")

---

## 第四步: 加载模型 (4bit 量化)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("🔄 加载 Qwen2.5-7B...")
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side='right'
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

print("✅ 模型加载完成!")

---

## 第五步: 配置 LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

---

## 第六步: 准备训练数据

In [ ]:
from datasets import load_dataset

MAX_LENGTH = 512

def format_example(example):
    text = f"### 指令:\n{example['instruction']}\n\n### 回答:\n{example['output']}"
    return {"text": text}

def tokenize(example):
    out = tokenizer(example["text"], max_length=MAX_LENGTH, truncation=True, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

dataset = load_dataset("json", data_files="nexify-landing/data/train.json", split="train")
dataset = dataset.map(format_example)
tokenized = dataset.map(tokenize, batched=True, remove_columns=["instruction", "output", "id", "text"])

print(f"✅ 数据准备完成: {len(tokenized)} 条")

---

## 第七步: 开始训练

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=100,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

print("🚀 开始训练...")
trainer.train()

---

## 第八步: 保存并下载模型

In [ ]:
model.save_pretrained("./output/lora_adapter")
tokenizer.save_pretrained("./output/lora_adapter")
print("✅ 模型已保存到 ./output/lora_adapter")

# 打包下载
import zipfile, os
from google.colab import files

with zipfile.ZipFile("lora_adapter.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("./output/lora_adapter"):
        for file in files:
            zf.write(os.path.join(root, file))

files.download("lora_adapter.zip")
print("📦 下载中...")

---

## 第九步: 推送到 HuggingFace

**先在下方单元格运行登录，再修改用户名并执行上传**

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # 弹出框输入 HF Token

In [ ]:
# ===== 修改这里为你的 HuggingFace 用户名 =====
HF_USERNAME = "你的HF用户名"  # 👈 例如: richard3153
MODEL_REPO = f"{HF_USERNAME}/qwen2.5-7b-ai-safety"

from huggingface_hub import create_repo, HfApi

create_repo(MODEL_REPO, exist_ok=True)
print(f"仓库: https://huggingface.co/{MODEL_REPO}")

api = HfApi()
api.upload_folder(
    folder_path="./output/lora_adapter",
    repo_id=MODEL_REPO,
    repo_type="model",
)

print(f"🎉 发布成功! https://huggingface.co/{MODEL_REPO}")